In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
df= pd.read_csv('/content/drive/MyDrive/Divyadharshiny-codebooster-internship-2026/messy_sales_data.csv')

In [11]:
import requests
import json
import matplotlib.pyplot as plt

import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")
print("pandas version : {pd.__version__}")
print("requests version : {requests.__version__}")
print("Loaded at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S)}")

Libraries imported successfully.
pandas version : {pd.__version__}
requests version : {requests.__version__}
Loaded at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S)}


In [12]:
raw_df = pd.read_csv('/content/drive/MyDrive/Divyadharshiny-codebooster-internship-2026/messy_sales_data.csv')
print(f"Rows : {raw_df.shape[0]}")
print(f"Columns  : {raw_df.shape[1]}")
print(f"Column names : {raw_df.columns.tolist()}")
print("\nFirst 5 row of raw data :")
raw_df.head()

Rows : 30
Columns  : 9
Column names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 5 row of raw data :


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [13]:
print("\n[1] MISSING VALUES per column:")
print(raw_df.isnull().sum())
print(f"\n[2] DUPLICATE ROWS : {raw_df.duplicated().sum()}")
print("\n[3] DATA TYPES :")
print(raw_df.dtypes)
print("\n[4] UNIQUE CATEGORIES :", raw_df['category'].dropna().unique().tolist())
print("[4] SAMPLE ORDER DATES :", raw_df['order_date'].unique()[:8].tolist())
print("[4] SAMPLE NAMES :", raw_df['customer_name'].dropna().unique()[:8].tolist())


[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS : 0

[3] DATA TYPES :
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES : ['Electronics', 'Accessories']
[4] SAMPLE ORDER DATES : ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']
[4] SAMPLE NAMES : ['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer']


In [14]:
df = raw_df.copy()
print(f"Working copy created with shape : {df.shape}")
print("raw_df is untouched. We can always reset with df = raw_df.copy()")

Working copy created with shape : (30, 9)
raw_df is untouched. We can always reset with df = raw_df.copy()


In [15]:
print(f"Total missing values before fix:{df.isnull().sum().sum()}")
df['customer_name'].fillna('Unknown customer',inplace=True)    #inplace=True  for updating the values directly to the df
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f"Filled  missing qquantity with median value:{median_qty}")
df['category'].fillna('Uncategorized',inplace=True)
df['product'].fillna('Unknown product',inplace=True)
print(df['product'].isnull().sum())
print(f"Total missing values after fix:{df.isnull().sum().sum()}")

Total missing values before fix:7
Filled  missing qquantity with median value:2.0
0
Total missing values after fix:0


In [16]:
print(f"Rows BEFORE deduplication : {len(df)}")
print(f"Duplicate rows found : {df.duplicated().sum()}")
duplicate_mark = df.duplicated(keep=False)
print("\n The duplicate rows (all copies shown) :")
print(df[duplicate_mark][['order_id', 'customer_name', 'product', 'order_date']].to_string(index=True))

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"\nRows AFTER deduplication : {len(df)}")
print(f"Rows removed : {len(raw_df) - len(df)}")

Rows BEFORE deduplication : 30
Duplicate rows found : 0

 The duplicate rows (all copies shown) :
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Rows AFTER deduplication : 30
Rows removed : 0


In [17]:
print("Sample dates before parrsing:")
print(df['order_date'].head(10).tolist())

df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')

nat_mask=df['order_date'].isnull()
df.loc[nat_mask,'order_date']=pd.to_datetime(raw_df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')

nat_count=df['order_date'].isnull().sum()
print(f"\nUnparseable dates remaining (NaT):{nat_count}")

df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['day_name']=df['order_date'].dt.strftime('%A')

print("\n sample dates after parsing:")
print(df[['order_date','year','month','month_name','day_name']].head(8).to_string())

Sample dates before parrsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']

Unparseable dates remaining (NaT):0

 sample dates after parsing:
  order_date  year  month month_name   day_name
0 2024-01-05  2024      1    January     Friday
1 2024-01-07  2024      1    January     Sunday
2 2024-01-08  2024      1    January     Monday
3 2024-01-10  2024      1    January  Wednesday
4 2024-01-05  2024      1    January     Friday
5 2024-01-07  2024      1    January     Sunday
6 2024-01-12  2024      1    January     Friday
7 2024-01-13  2024      1    January   Saturday


In [18]:
print("Name BEFOR standardization (sample):")
print(df['customer_name'].unique()[:8].tolist())
df['customer_name'] = df['customer_name'].str.strip().str.title()
print("\nName AFTER standardization (sample):")
print(df['customer_name'].unique()[:8].tolist())

Name BEFOR standardization (sample):
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Unknown customer', 'Ananya Das']

Name AFTER standardization (sample):
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das']


In [19]:
wrong_mask = (df['product'] == 'Keyboard') & (df['category'] == 'Electronics')
print(f"Rows to fix : {wrong_mask.sum()}")
print("Before fix :")
print(df[wrong_mask][['order_id', 'product', 'category']].to_string(index=False))
print("\n All unique categories now :", sorted(df['category'].unique().tolist()))

Rows to fix : 1
Before fix :
 order_id  product    category
     1024 Keyboard Electronics

 All unique categories now : ['Accessories', 'Electronics', 'Uncategorized']


In [20]:
df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce').astype(int)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors = 'coerce')
df['revenue'] = df['quantity'] * df['unit_price']
print("Revenue column created successfully.")
print("\n sample:quantity x unit_price = revenue")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))

Revenue column created successfully.

 sample:quantity x unit_price = revenue
   customer_name         product  quantity  unit_price  revenue
    Ramesh Kumar          Laptop         2       45000    90000
      Priya Nair Unknown product         1       15000    15000
      Amit Verma        Keyboard         3        1200     3600
    Sunita Patel         Monitor         2       22000    44000
    Ramesh Kumar          Laptop         2       45000    90000
     Kiran Mehta           Mouse        10         800     8000
    Deepak Singh      Headphones         2        3500     7000
Unknown Customer          Webcam         1        2500     2500


In [21]:
# ============================================================
# CELL 11 — Post-Cleaning Validation Report
# ============================================================


# Calculate the data quality score
# We check 5 things: no missing values, no duplicates, no date nulls, no revenue nulls
# Each passing check contributes 20 points (5 checks × 20 = 100)
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(df)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")



  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 0
  Duplicates      : 0
  Date nulls      : 0
  Revenue nulls   : 0
  Columns total   : 14
  DATA QUALITY SCORE : 100/100
  DATA IS CLEAN      : True

  All checks passed. Data is ready for analysis.


In [22]:
output_filename = 'clean_sales_data.csv'
df.to_csv(output_filename, index=False)

print(f"Cleaned data saved to: {output_filename}")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nCleaned data saved to: {output_filename}")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(" EXTRACT  -> messy_sales_data.csv loaded")
print(" TRANSFORM -> 6 issues fixed (nulls, dupes, dates, names, categories, types)")
print("LOAD  -> clean_sales_data.csv saved")

Cleaned data saved to: clean_sales_data.csv
Final dataset: 30 rows x 14 columns

Cleaned data saved to: clean_sales_data.csv
Final dataset: 30 rows x 14 columns
 EXTRACT  -> messy_sales_data.csv loaded
 TRANSFORM -> 6 issues fixed (nulls, dupes, dates, names, categories, types)
LOAD  -> clean_sales_data.csv saved


In [35]:
SERP_API_KEY = 'e3fbe4dc1a0be0fa382f057c9e8806aa4058c66db4e4415df93a00f974db40a7'
SERP_URL='https://serpapi.com/search.json'
SEARCH_QUERY='Data Engineer India'
print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"Search query : {search_query}")

SerpAPI Key  : Set (live data)
Search query : Data Engineer India


In [36]:
def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(SERP_URL, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if SERP_API_KEY != 'SERP_API_KEY ':
    print(f"Fetching job listings for: '{SEARCH_QUERY}'")
    job_records = fetch_jobs(SEARCH_QUERY, SERP_API_KEY)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")

Fetching job listings for: 'Data Engineer India'
  Page 1: fetched 10 jobs
  Page 2: fetched 10 jobs
Total jobs fetched: 20


In [37]:
output_filename='clean_data_engineering.csv'

# Convert job_records (list of dicts) to a DataFrame
job_df = pd.DataFrame(job_records)

# Save the DataFrame to CSV
job_df.to_csv(output_filename,index=False)

print(f"Cleaned job data saved to: {output_filename}")
print(f"Final dataset: {job_df.shape[0]} rows x {job_df.shape[1]} columns")

Cleaned job data saved to: clean_data_engineering.csv
Final dataset: 20 rows x 7 columns
